In [ ]:
# =============================================================================
# Multi-Agent SLM Pipeline — Kaggle Script
# Supports: DeepSeek-R1-1.5B | Gemma-2-9B | Mistral-7B | Qwen2.5-7B
# =============================================================================

## Cell 1 — Install dependencies
Run this cell first, then restart the kernel before continuing.

In [ ]:
!pip install -q \
    "transformers>=4.45.0" \
"accelerate>=0.27.0" \
"bitsandbytes>=0.43.0" \
"datasets>=2.20.0" \
"sentencepiece>=0.1.99" \
  "tiktoken>=0.6.0" \
   "huggingface_hub>=0.22.0"

# Versions explained:
#   transformers>=4.45.0  — Qwen2.5 + DeepSeek-R1-Distill-Qwen need >=4.37;
#                           Mistral-7B-v0.3 and Gemma-2 need >=4.42;
#                           4.45 is the safe unified floor.
#   accelerate>=0.27.0    — required for device_map="auto" + quantization.
#   bitsandbytes>=0.43.0  — stable 4-bit NF4 on CUDA 12.x (Kaggle T4/P100).
#   sentencepiece         — Mistral v0.3 tokenizer hard-requires it.
#   tiktoken              — Qwen2.5 and DeepSeek-R1-Distill-Qwen (Qwen-based)
#                           require it; missing = tokenizer crash on load.
#   huggingface_hub       — for HF_TOKEN auth (needed for gated Gemma 2).


# =============================================================================
# ██████╗  ██████╗ ███╗   ██╗███████╗██╗ ██████╗
# ██╔════╝██╔═══██╗████╗  ██║██╔════╝██║██╔════╝
# ██║     ██║   ██║██╔██╗ ██║█████╗  ██║██║  ███╗
# ██║     ██║   ██║██║╚██╗██║██╔══╝  ██║██║   ██║
# ╚██████╗╚██████╔╝██║ ╚████║██║     ██║╚██████╔╝
#  ╚═════╝ ╚═════╝ ╚═╝  ╚═══╝╚═╝     ╚═╝ ╚═════╝
# Edit the values in this block to control the experiment.
# =============================================================================

## Cell 2 — Configuration

In [1]:
# ── Model ─────────────────────────────────────────────────────────────────────
# Pick ONE of the keys below and set it as ACTIVE_MODEL.
#
#   "deepseek"  →  deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B
#                  • ~1.5 B params, fits in fp16 easily.
#                  • Outputs <think>...</think> reasoning blocks — stripped
#                    automatically before answer extraction.
#                  • temperature=0.0 (greedy) causes repetition loops per
#                    official docs; overridden to 0.6 / do_sample=True below.
#                  • Not gated — no HF token needed.
#
#   "gemma"     →  google/gemma-2-9b-it
#                  • ~9 B params, loaded in 4-bit NF4 (~5-6 GB VRAM on T4).
#                  • GATED MODEL — you must:
#                      1. Accept the licence at huggingface.co/google/gemma-2-9b-it
#                      2. Add your HF token to Kaggle Secrets as HF_TOKEN
#                         (Notebook → Add-ons → Secrets).
#                  • Native bfloat16 but T4 only supports float16 — set to fp16.
#
#   "mistral"   →  mistralai/Mistral-7B-Instruct-v0.3
#                  • ~7 B params, loaded in 4-bit NF4 (~4-5 GB VRAM).
#                  • Requires sentencepiece (included in install cell above).
#                  • Requires transformers>=4.42.0.
#                  • Not gated — no HF token needed.
#
#   "qwen"      →  Qwen/Qwen2.5-7B-Instruct
#                  • ~7 B params, loaded in 4-bit NF4 (~4-5 GB VRAM).
#                  • Requires tiktoken (included in install cell above).
#                  • Requires transformers>=4.37.0.
#                  • Not gated — no HF token needed.

ACTIVE_MODEL = "deepseek"

MODEL_CONFIGS = {
    "deepseek": {
        "name":         "deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B",
        "dtype":        "float16",
        "load_in_4bit": False,
        # DeepSeek-R1 official docs explicitly warn that temperature=0 (greedy)
        # causes endless repetition. Recommended range: 0.5–0.7.
        "temperature":  0.6,
        "do_sample":    True,
        "top_p":        0.95,
        "gated":        False,
    },
    "gemma": {
        "name":         "google/gemma-2-9b-it",
        "dtype":        "float16",   # T4 does NOT support bfloat16 natively
        "load_in_4bit": True,
        "temperature":  0.0,
        "do_sample":    False,
        "top_p":        1.0,
        "gated":        True,        # requires HF_TOKEN in Kaggle Secrets
    },
    "mistral": {
        "name":         "mistralai/Mistral-7B-Instruct-v0.3",
        "dtype":        "float16",
        "load_in_4bit": True,
        "temperature":  0.0,
        "do_sample":    False,
        "top_p":        1.0,
        "gated":        False,
    },
    "qwen": {
        "name":         "Qwen/Qwen2.5-7B-Instruct",
        "dtype":        "float16",   # T4 does NOT support bfloat16 natively
        "load_in_4bit": True,
        "temperature":  0.0,
        "do_sample":    False,
        "top_p":        1.0,
        "gated":        False,
    },
}

# ── Dataset ───────────────────────────────────────────────────────────────────
DATASET_NAME    = "svamp"    # "gsm8k" | "svamp" | "math"
DATASET_SPLIT   = "train"
MAX_SAMPLES     = 100
SEED            = 42

# ── Pipeline ──────────────────────────────────────────────────────────────────
PIPELINE_TYPE   = "planner_solver_verifier"  # "solver" | "planner_solver" | "planner_solver_verifier"
COMM_MODE       = "structured"               # "natural" | "constrained" | "structured"
COMM_MAX_TOKENS = 120
COMM_NUM_STEPS  = 5

# ── Generation defaults (overridden per-model from MODEL_CONFIGS above) ───────
MAX_NEW_TOKENS  = 512

# ── Output ────────────────────────────────────────────────────────────────────
OUTPUT_DIR = f"/kaggle/working/{ACTIVE_MODEL}_{DATASET_NAME}"


# =============================================================================
# Imports
# =============================================================================

## Cell 3 — Imports & HF auth

In [2]:
from __future__ import annotations

import json
import os
import random
import re
from dataclasses import dataclass
from pathlib import Path
from typing import Any

import numpy as np
import torch
from datasets import load_dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
)


# =============================================================================
# HuggingFace Authentication (needed for gated models like Gemma 2)
# =============================================================================

In [3]:
def _hf_login_if_needed(gated: bool) -> None:
    if not gated:
        return
    try:
        from kaggle_secrets import UserSecretsClient
        token = UserSecretsClient().get_secret("HF_TOKEN")
        from huggingface_hub import login
        login(token=token)
        print("✓ Logged in to HuggingFace via Kaggle secret HF_TOKEN")
    except Exception as e:
        print(
            f"[WARNING] Could not read HF_TOKEN from Kaggle Secrets: {e}\n"
            "If this is a gated model (e.g. Gemma 2), the download will fail.\n"
            "Add your HF token under Notebook → Add-ons → Secrets → HF_TOKEN."
        )


# =============================================================================
# Utilities
# =============================================================================

In [4]:
def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def ensure_dir(path: str) -> None:
    Path(path).mkdir(parents=True, exist_ok=True)


def save_json(path: str, payload: Any) -> None:
    with open(path, "w", encoding="utf-8") as f:
        json.dump(payload, f, indent=2, ensure_ascii=False)


GSM8K_ANSWER_RE = re.compile(r"####\s*([-+]?\d[\d,.]*)")

def extract_gsm8k_answer(answer_text: str) -> str:
    match = GSM8K_ANSWER_RE.search(answer_text)
    if match:
        return match.group(1).replace("$", "").replace(",", "").strip()
    tail = answer_text.split("####")[-1].strip()
    return tail.replace("$", "").replace(",", "").strip()


# =============================================================================
# Model
# =============================================================================

In [5]:
class HuggingFaceTextGenerator:
    def __init__(
        self,
        model_name:     str,
        dtype_str:      str,
        load_in_4bit:   bool,
        max_new_tokens: int,
        temperature:    float,
        top_p:          float,
        do_sample:      bool,
    ):
        self.max_new_tokens = max_new_tokens
        self.temperature    = temperature
        self.top_p          = top_p
        self.do_sample      = do_sample

        # T4 does NOT support bfloat16; always fall back to float16.
        dtype_map = {"float16": torch.float16, "bfloat16": torch.float16}
        dtype = dtype_map.get(dtype_str, torch.float16)

        quant_config = None
        if load_in_4bit:
            quant_config = BitsAndBytesConfig(
                load_in_4bit=True,
                bnb_4bit_quant_type="nf4",
                bnb_4bit_use_double_quant=True,
                # compute dtype must also be float16 on T4
                bnb_4bit_compute_dtype=torch.float16,
            )

        self.tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True)
        if self.tokenizer.pad_token is None:
            self.tokenizer.pad_token = self.tokenizer.eos_token

        model_kwargs: dict[str, Any] = {"trust_remote_code": True}
        if quant_config:
            model_kwargs["quantization_config"] = quant_config
        else:
            model_kwargs["torch_dtype"] = dtype

        self.model = AutoModelForCausalLM.from_pretrained(
            model_name,
            device_map="auto",   # spreads across both T4s if T4x2 is selected
            **model_kwargs,
        )
        self.model.eval()
        print(f"✓ Loaded {model_name} | 4bit={load_in_4bit} | dtype={dtype_str}")

    @torch.inference_mode()
    def generate(self, prompt: str) -> dict[str, Any]:
        max_length = getattr(self.model.config, "max_position_embeddings", 2048)
        encoded = self.tokenizer(
            prompt, return_tensors="pt", truncation=True, max_length=max_length
        )
        encoded = {k: v.to(self.model.device) for k, v in encoded.items()}

        gen_kwargs: dict[str, Any] = dict(
            max_new_tokens=self.max_new_tokens,
            do_sample=self.do_sample,
            pad_token_id=self.tokenizer.pad_token_id,
            eos_token_id=self.tokenizer.eos_token_id,
        )
        # Only pass temperature / top_p when actually sampling.
        # Passing temperature != 1.0 with do_sample=False raises a
        # UserWarning (and in newer transformers versions, an error).
        if self.do_sample:
            gen_kwargs["temperature"] = self.temperature
            gen_kwargs["top_p"]       = self.top_p

        output = self.model.generate(**encoded, **gen_kwargs)

        input_len      = encoded["input_ids"].shape[1]
        completion_ids = output[0][input_len:]
        text = self.tokenizer.decode(completion_ids, skip_special_tokens=True).strip()
        return {
            "text":              text,
            "prompt_tokens":     int(input_len),
            "completion_tokens": int(completion_ids.shape[0]),
        }


# =============================================================================
# Communication
# =============================================================================

In [6]:
class CommunicationTransform:
    def __call__(self, planner_text: str) -> str:
        raise NotImplementedError


class NaturalCommunication(CommunicationTransform):
    def __call__(self, planner_text: str) -> str:
        return planner_text.strip()


class ConstrainedCommunication(CommunicationTransform):
    def __init__(self, max_tokens: int, tokenizer=None):
        self.max_tokens = max_tokens
        self.tokenizer  = tokenizer

    def __call__(self, planner_text: str) -> str:
        if self.tokenizer is not None:
            ids = self.tokenizer.encode(planner_text.strip(), add_special_tokens=False)
            return self.tokenizer.decode(ids[: self.max_tokens], skip_special_tokens=True).strip()
        words = planner_text.strip().split()
        return " ".join(words[: self.max_tokens])


class StructuredCommunication(CommunicationTransform):
    def __init__(self, max_tokens: int, num_steps: int, tokenizer=None):
        self.max_tokens = max_tokens
        self.num_steps  = num_steps
        self.tokenizer  = tokenizer

    # Matches: "1.", "1)", "Step 1:", "- text", "• text", "* text"
    _STEP_RE = re.compile(
        r"^(\d+[.):]|step\s*\d+[.:]?|[-•*])\s+\S",
        flags=re.IGNORECASE,
    )
    _STRIP_PREFIX_RE = re.compile(
        r"^(\d+[.):]|step\s*\d+[.:]?|[-•*])\s+",
        flags=re.IGNORECASE,
    )

    def __call__(self, planner_text: str) -> str:
        lines = [line.strip() for line in planner_text.splitlines() if line.strip()]
        candidate_steps = []

        # Check if first line is a step without a number prefix (because prompt pre-filled "1.")
        if lines and not self._STEP_RE.match(lines[0]):
            candidate_steps.append(lines[0])  # Add it manually as step 1

        for line in lines:
            if self._STEP_RE.match(line):
                cleaned = self._STRIP_PREFIX_RE.sub("", line).strip()
                candidate_steps.append(cleaned)

        if not candidate_steps:
            # Fall back: sentence split, keep non-trivial sentences (>15 chars)
            sentences = re.split(r"(?<=[.!?])\s+", planner_text.strip())
            candidate_steps = [s.strip() for s in sentences if len(s.strip()) > 15]

        selected = candidate_steps[: self.num_steps]

        # Use plain "1." numbering — more natural and fewer tokens than [STEP_X]
        structured = "\n".join(f"{i + 1}. {step}" for i, step in enumerate(selected))

        if self.tokenizer is not None:
            ids = self.tokenizer.encode(structured, add_special_tokens=False)
            return self.tokenizer.decode(ids[: self.max_tokens], skip_special_tokens=True).strip()
        words = structured.split()
        return " ".join(words[: self.max_tokens])


def build_communication(mode: str, max_tokens: int, num_steps: int, tokenizer=None) -> CommunicationTransform:
    mode = mode.lower()
    if mode == "natural":
        return NaturalCommunication()
    if mode == "constrained":
        return ConstrainedCommunication(max_tokens=max_tokens, tokenizer=tokenizer)
    if mode == "structured":
        return StructuredCommunication(max_tokens=max_tokens, num_steps=num_steps, tokenizer=tokenizer)
    raise ValueError(f"Unsupported communication mode: {mode!r}")


# =============================================================================
# Agents
# =============================================================================

In [7]:
@dataclass
class AgentOutput:
    text:              str
    prompt_tokens:     int
    completion_tokens: int


class BaseAgent:
    def __init__(self, model: HuggingFaceTextGenerator):
        self.model = model

    def _call(self, prompt: str) -> AgentOutput:
        out = self.model.generate(prompt)
        return AgentOutput(
            text=out["text"],
            prompt_tokens=out["prompt_tokens"],
            completion_tokens=out["completion_tokens"],
        )


class PlannerAgent(BaseAgent):
    def run(self, question: str) -> AgentOutput:
        prompt = (
            "You are a planning agent. Your ONLY job is to list steps.\n"
            "Rules:\n"
            "- Output ONLY a numbered list. Nothing else.\n"
            "- Maximum 4 steps.\n"
            "- Do NOT compute any numbers.\n"
            "- Do NOT write explanations, examples, or solutions.\n"
            "- Stop immediately after listing the steps.\n"
            "- First step must identify which numbers in the problem are relevant "
            "to the question and which are distractors to ignore.\n\n"
            f"Question: {question}\n\n"
            "Steps:\n"
            "1."
        )
        return self._call(prompt)


class SolverAgent(BaseAgent):
    def run(self, question: str, communication: str | None = None) -> AgentOutput:
        plan_section = (
            f"Step-by-step plan:\n{communication}\n\n"
            "Follow the above plan step by step.\n\n"
        ) if communication else ""
        prompt = (
            "You are a solver agent. Solve the problem carefully and give the final answer.\n"
            "End with a line exactly in this format: FINAL_ANSWER: <answer>\n\n"
            f"{plan_section}"
            f"Question: {question}\n\n"
            "Solution:"
        )
        return self._call(prompt)


class VerifierAgent(BaseAgent):
    def run(self, question: str, draft_answer: str) -> AgentOutput:
        prompt = (
            "You are a verification agent. Check the proposed solution carefully.\n"
            "If the reasoning and final answer are correct, output exactly:\n"
            "  VERIFICATION: correct\n"
            "  FINAL_ANSWER: <same answer as proposed>\n"
            "If you find an error, provide the corrected reasoning and output:\n"
            "  VERIFICATION: incorrect\n"
            "  FINAL_ANSWER: <corrected answer>\n"
            "You MUST always end your response with a FINAL_ANSWER: line containing "
            "only the answer value, nothing else.\n\n"
            f"Question: {question}\n\n"
            f"Proposed solution:\n{draft_answer}\n\n"
            "Verification:"
        )
        return self._call(prompt)


FINAL_ANSWER_RE  = re.compile(r"FINAL_ANSWER\s*:\s*(.+)", re.IGNORECASE)
_NUMBER_RE_AGENT = re.compile(r"[-+]?\d[\d,.]*")
# DeepSeek-R1 wraps all chain-of-thought in <think>...</think> before the answer.
_THINK_RE        = re.compile(r"<think>.*?</think>", re.DOTALL | re.IGNORECASE)


def extract_final_answer(text: str) -> str:
    # Strip DeepSeek-R1 <think> blocks before searching for FINAL_ANSWER.
    # For other models this is a no-op (no <think> tag present).
    clean = _THINK_RE.sub("", text).strip()

    match = FINAL_ANSWER_RE.search(clean)
    if match:
        return match.group(1).strip()
    numbers = _NUMBER_RE_AGENT.findall(clean)
    if numbers:
        return numbers[-1].replace(",", "")
    lines = [line.strip() for line in clean.splitlines() if line.strip()]
    return lines[-1] if lines else clean.strip()


# =============================================================================
# Metrics
# =============================================================================

In [9]:
def normalize_answer(text: str) -> str:
    text = text.strip()
    boxed = re.search(r"\\boxed\{(.+)\}", text, re.DOTALL)
    if boxed:
        text = boxed.group(1)
    text = text.strip().lower()
    text = re.sub(r"[$,%]", "", text)
    text = re.sub(r"\s+", " ", text)
    text = text.rstrip(".")
    text = re.sub(r"\.0+$", "", text)
    text = re.sub(r"(\d),(\d)", r"\1\2", text)
    return text.strip()


_NUMBER_RE_METRIC = re.compile(r"[-+]?\d+(?:\.\d+)?")
_FRAC_RE          = re.compile(r"\\frac\{([^{}]+)\}\{([^{}]+)\}")


def _try_frac(text: str) -> float | None:
    m = _FRAC_RE.match(text.strip())
    if m:
        try:
            return float(m.group(1)) / float(m.group(2))
        except (ValueError, ZeroDivisionError):
            return None
    return None


def _try_float(text: str) -> float | None:
    try:
        return float(text)
    except ValueError:
        return None


def is_correct(prediction: str, gold: str) -> bool:
    pred_norm = normalize_answer(prediction)
    gold_norm = normalize_answer(gold)
    if pred_norm == gold_norm:
        return True
    pred_f = _try_float(pred_norm)
    gold_f = _try_float(gold_norm)
    if pred_f is not None and gold_f is not None:
        return abs(pred_f - gold_f) < 1e-6
    pred_frac = _try_frac(pred_norm)
    gold_frac = _try_frac(gold_norm)
    if pred_frac is not None and gold_frac is not None:
        return abs(pred_frac - gold_frac) < 1e-6
    if pred_frac is not None and gold_f is not None:
        return abs(pred_frac - gold_f) < 1e-6
    if gold_frac is not None and pred_f is not None:
        return abs(pred_f - gold_frac) < 1e-6
    numbers = _NUMBER_RE_METRIC.findall(pred_norm)
    if numbers:
        last_f = _try_float(numbers[-1])
        if last_f is not None and gold_f is not None:
            return abs(last_f - gold_f) < 1e-6
    return False


def summarize_results(rows: list[dict[str, Any]]) -> dict[str, Any]:
    total      = len(rows)
    correct    = sum(1 for row in rows if row["correct"])
    token_cost = sum(int(row["token_cost"]) for row in rows)
    avg_tokens = token_cost / total if total else 0.0
    accuracy   = correct / total if total else 0.0
    return {
        "model":                    MODEL_CONFIGS[ACTIVE_MODEL]["name"],
        "pipeline":                 PIPELINE_TYPE,
        "communication":            COMM_MODE,
        "dataset":                  DATASET_NAME,
        "num_examples":             total,
        "num_correct":              correct,
        "accuracy":                 round(accuracy, 4),
        "total_token_cost":         token_cost,
        "avg_token_cost":           round(avg_tokens, 2),
        "accuracy_per_1000_tokens": round((accuracy / avg_tokens) * 1000, 6) if avg_tokens else 0.0,
    }


# =============================================================================
# Data
# =============================================================================

In [10]:
def _extract_boxed_answer(solution: str) -> str:
    idx = solution.rfind(r"\boxed{")
    if idx == -1:
        return solution.strip()
    start = idx + len(r"\boxed{")
    depth = 1
    pos   = start
    while pos < len(solution) and depth > 0:
        if solution[pos] == "{":
            depth += 1
        elif solution[pos] == "}":
            depth -= 1
        pos += 1
    return solution[start: pos - 1].strip()


def load_dataset_records(name: str, split: str, max_samples: int, seed: int) -> list[dict[str, str]]:
    if name == "gsm8k":
        ds = load_dataset("gsm8k", "main", split=split).shuffle(seed=seed)
        rows = []
        for item in ds.select(range(min(max_samples, len(ds)))):
            rows.append({
                "question": str(item["question"]),
                "answer":   extract_gsm8k_answer(str(item["answer"])),
            })
        return rows

    if name == "svamp":
        ds = load_dataset("ChilleD/SVAMP", split="train").shuffle(seed=seed)
        rows = []
        for item in ds.select(range(min(max_samples, len(ds)))):
            body     = str(item.get("Body", "")).strip()
            question = str(item.get("Question", "")).strip()
            rows.append({
                "question": f"{body} {question}".strip(),
                "answer":   str(item["Answer"]).strip(),
            })
        return rows

    if name == "math":
        ds = load_dataset("nlile/hendrycks-MATH-benchmark", split=split).shuffle(seed=seed)
        rows = []
        for item in ds.select(range(min(max_samples, len(ds)))):
            raw_answer = item.get("answer") or ""
            answer = raw_answer.strip() if raw_answer.strip() else _extract_boxed_answer(str(item["solution"]))
            rows.append({
                "question": str(item["problem"]),
                "answer":   answer,
            })
        return rows

    raise ValueError(f"Unsupported dataset: {name!r}. Choose from: gsm8k, svamp, math")


# =============================================================================
# Pipeline
# =============================================================================

In [11]:
class SolverOnlyPipeline:
    def __init__(self, model):
        self.solver = SolverAgent(model)

    def run(self, example: dict) -> dict:
        solver = self.solver.run(example["question"])
        pred   = extract_final_answer(solver.text)
        cost   = solver.prompt_tokens + solver.completion_tokens
        return {
            "planner_text":    None,
            "communication":   None,
            "solver_text":     solver.text,
            "verifier_text":   None,
            "prediction":      pred,
            "correct":         is_correct(pred, example["answer"]),
            "token_cost":      cost,
            "planner_tokens":  0,
            "solver_tokens":   cost,
            "verifier_tokens": 0,
        }


class PlannerSolverPipeline:
    def __init__(self, model, comm):
        self.planner = PlannerAgent(model)
        self.solver  = SolverAgent(model)
        self.comm    = comm

    def run(self, example: dict) -> dict:
        planner = self.planner.run(example["question"])
        comm    = self.comm(planner.text)
        solver  = self.solver.run(example["question"], comm)
        pred    = extract_final_answer(solver.text)
        cost    = (planner.prompt_tokens + planner.completion_tokens
                   + solver.prompt_tokens + solver.completion_tokens)
        return {
            "planner_text":    planner.text,
            "communication":   comm,
            "solver_text":     solver.text,
            "verifier_text":   None,
            "prediction":      pred,
            "correct":         is_correct(pred, example["answer"]),
            "token_cost":      cost,
            "planner_tokens":  planner.prompt_tokens + planner.completion_tokens,
            "solver_tokens":   solver.prompt_tokens  + solver.completion_tokens,
            "verifier_tokens": 0,
        }


class PlannerSolverVerifierPipeline:
    def __init__(self, model, comm):
        self.planner  = PlannerAgent(model)
        self.solver   = SolverAgent(model)
        self.verifier = VerifierAgent(model)
        self.comm     = comm

    def run(self, example: dict) -> dict:
        planner  = self.planner.run(example["question"])
        comm     = self.comm(planner.text)
        solver   = self.solver.run(example["question"], comm)
        verifier = self.verifier.run(example["question"], solver.text)
        pred     = extract_final_answer(verifier.text)
        cost     = sum([
            planner.prompt_tokens,  planner.completion_tokens,
            solver.prompt_tokens,   solver.completion_tokens,
            verifier.prompt_tokens, verifier.completion_tokens,
        ])
        return {
            "planner_text":    planner.text,
            "communication":   comm,
            "solver_text":     solver.text,
            "verifier_text":   verifier.text,
            "prediction":      pred,
            "correct":         is_correct(pred, example["answer"]),
            "token_cost":      cost,
            "planner_tokens":  planner.prompt_tokens  + planner.completion_tokens,
            "solver_tokens":   solver.prompt_tokens   + solver.completion_tokens,
            "verifier_tokens": verifier.prompt_tokens + verifier.completion_tokens,
        }


def build_pipeline(pipeline_type: str, model, comm):
    if pipeline_type == "solver":
        return SolverOnlyPipeline(model)
    if pipeline_type == "planner_solver":
        return PlannerSolverPipeline(model, comm)
    if pipeline_type == "planner_solver_verifier":
        return PlannerSolverVerifierPipeline(model, comm)
    raise ValueError(f"Unsupported pipeline type: {pipeline_type!r}")


# =============================================================================
# Main
# =============================================================================

## Cell 4 — Run

In [12]:
def main():
    set_seed(SEED)
    ensure_dir(OUTPUT_DIR)

    mcfg = MODEL_CONFIGS[ACTIVE_MODEL]

    print(f"\n{'='*60}")
    print(f"  Model    : {mcfg['name']}")
    print(f"  Pipeline : {PIPELINE_TYPE}")
    print(f"  Comm     : {COMM_MODE}")
    print(f"  Dataset  : {DATASET_NAME}  ({MAX_SAMPLES} samples)")
    print(f"  Sampling : do_sample={mcfg['do_sample']}  temperature={mcfg['temperature']}  top_p={mcfg['top_p']}")
    print(f"{'='*60}\n")

    # Authenticate if the model is gated (e.g. Gemma 2)
    _hf_login_if_needed(mcfg["gated"])

    # Load model
    model = HuggingFaceTextGenerator(
        model_name     = mcfg["name"],
        dtype_str      = mcfg["dtype"],
        load_in_4bit   = mcfg["load_in_4bit"],
        max_new_tokens = MAX_NEW_TOKENS,
        temperature    = mcfg["temperature"],
        top_p          = mcfg["top_p"],
        do_sample      = mcfg["do_sample"],
    )

    # Load dataset
    dataset = load_dataset_records(DATASET_NAME, DATASET_SPLIT, MAX_SAMPLES, SEED)
    print(f"Loaded {len(dataset)} examples from {DATASET_NAME}\n")

    # Build communication + pipeline
    comm     = build_communication(COMM_MODE, COMM_MAX_TOKENS, COMM_NUM_STEPS, model.tokenizer)
    pipeline = build_pipeline(PIPELINE_TYPE, model, comm)

    # Run
    rows = []
    for idx, example in enumerate(dataset):
        try:
            result = pipeline.run(example)
        except Exception as exc:
            print(f"[{idx + 1}/{len(dataset)}] ERROR: {exc}")
            result = {
                "planner_text":    None,
                "communication":   None,
                "solver_text":     None,
                "verifier_text":   None,
                "prediction":      "",
                "correct":         False,
                "token_cost":      0,
                "planner_tokens":  0,
                "solver_tokens":   0,
                "verifier_tokens": 0,
                "error":           str(exc),
            }

        row = {
            "index":       idx,
            "question":    example["question"],
            "gold_answer": example["answer"],
            **result,
        }
        rows.append(row)
        status = "✓" if row["correct"] else "✗"
        print(
            f"[{idx + 1:>3}/{len(dataset)}] {status}  "
            f"pred={row['prediction']!r:15}  "
            f"gold={row['gold_answer']!r:10}  "
            f"tokens={row['token_cost']}"
        )

        if (idx + 1) % 10 == 0 or (idx + 1) == len(dataset):
            save_json(f"{OUTPUT_DIR}/predictions.json", rows)

    # Summary
    summary = summarize_results(rows)
    save_json(f"{OUTPUT_DIR}/summary.json", summary)

    print(f"\n{'='*60}  SUMMARY  {'='*60}")
    for key, value in summary.items():
        print(f"  {key:<35}: {value}")
    print("=" * 130)
    print(f"\nResults saved to {OUTPUT_DIR}/")


main()


  Model    : deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B
  Pipeline : planner_solver_verifier
  Comm     : structured
  Dataset  : svamp  (100 samples)
  Sampling : do_sample=True  temperature=0.6  top_p=0.95



config.json:   0%|          | 0.00/679 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


...

data/train-00000-of-00001.parquet:   0%|          | 0.00/111k [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/54.8k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/700 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/300 [00:00<?, ? examples/s]

Loaded 100 examples from svamp

[  1/100] ✓  pred='21.'            gold='21'        tokens=2493
[  2/100] ✗  pred='The reasoning and final answer are correct.'  gold='8'         tokens=1075
[  3/100] ✗  pred='50.'            gold='6'         tokens=2375
[  4/100] ✓  pred='4.'             gold='4'         tokens=2486
[  5/100] ✓  pred='41'             gold='41'        tokens=2463
[  6/100] ✗  pred='Final Answer: <same answer as proposed>'  gold='28'        tokens=1356
[  7/100] ✓  pred='2'              gold='2'         tokens=1867
[  8/100] ✗  pred='48'             gold='65'        tokens=2179
[  9/100] ✓  pred='21'             gold='21'        tokens=1636
[ 10/100] ✗  pred='Correct'        gold='112'       tokens=1355
[ 11/100] ✓  pred='1'              gold='1'         tokens=1767
[ 12/100] ✓  pred='3'              gold='3'         tokens=2502
[ 13/100] ✗  pred='14'             gold='52'        tokens=2421
[ 14/100] ✓  pred='4'              gold='4'         tokens=1430
[ 15/100] ✗  pre